# Synthetic Data Generation 可視化流程

這份 Notebook 對照 `synthetic_data_generation.md`，逐步展示「假資料生成」與「表格渲染檢測」每一步實際會產生什麼內容，方便你調參。


## 0) 初始化與參數

> 你可以先改 `SEED` / `TEMPLATE_IMAGE` 來觀察不同結果。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

from score_reader.dataset.generator.ground_truth_generator import GroundTruthGenerator
from score_reader.dataset.generator.sheet_renderer import SheetRenderer

SEED = 42
TEMPLATE_IMAGE = Path("sample.png")
assert TEMPLATE_IMAGE.exists(), f"找不到模板圖片: {TEMPLATE_IMAGE}"

gt = GroundTruthGenerator()
renderer = SheetRenderer()


## 1) 產生 ground truth（第一大項）

這一步先不畫圖，直接印出「預期要填進去」的假資料內容。


In [ ]:
sheet = gt.generate(image_id="demo_notebook", seed=SEED)
print(f"image_id={sheet.image_id}, seed={sheet.seed}")
print(f"targets={len(sheet.targets)}")


In [ ]:
# 展示第 1 位選手（target_1）每一個 end 的細節：
t0 = sheet.targets[0]
print("target_id:", t0.target_id)
print("name:", t0.name)
print("club:", t0.club)
print("\n[每局內容]")
for i, end in enumerate(t0.rounds, start=1):
    arrows = [a.value for a in end.arrows]
    print(f"end {i}: arrows={arrows}, subtotal={end.subtotal}, cumulative={end.cumulative}")
print("\n[總結]")
print("x_count=", t0.x_count)
print("x_plus_ten_count=", t0.x_plus_ten_count)
print("total=", t0.total)


## 2) Rendering into template（第二大項）

以下 cells 專注在你指定的重點：
- 檢測到的行（lines）用**淺綠色**標註。
- 檢測到的選手區域用**紅色框**。
- 每一個小框（cell）用**淺藍色**標註。


In [ ]:
img = Image.open(TEMPLATE_IMAGE).convert("RGB")
gray = img.convert("L")
arr = np.array(gray)
h, w = arr.shape
print(f"template size: {w} x {h}")


In [ ]:
# 2.1 Detect table lines
# 直接對照 SheetRenderer._detect_cells 內部邏輯
dark_cols = [x for x in range(w) if (arr[:, x] < 110).sum() > h * 0.20]
dark_rows = [y for y in range(h) if (arr[y, :] < 110).sum() > w * 0.20]

v_lines = renderer._merge_lines(dark_cols)
h_lines = renderer._merge_lines(dark_rows)

print("raw dark cols:", len(dark_cols), "=> merged vertical lines:", len(v_lines))
print("raw dark rows:", len(dark_rows), "=> merged horizontal lines:", len(h_lines))
print("first few v_lines:", v_lines[:12])
print("first few h_lines:", h_lines[:12])


In [ ]:
# 視覺化：檢測到的行（淺綠）
fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(img)
for x in v_lines:
    ax.axvline(x=x, color="#90EE90", linewidth=1.5)  # light green
for y in h_lines:
    ax.axhline(y=y, color="#90EE90", linewidth=1.5)  # light green
ax.set_title("2.1 Detected table lines (light green)")
ax.axis("off")
plt.show()


In [ ]:
# 2.2 Detect 4 athlete scoring regions
regions = renderer._detect_target_regions(v_lines, h_lines)
print("detected regions:", len(regions))
for i, (l, t, r, b) in enumerate(regions, start=1):
    print(f"region {i}: left={l}, top={t}, right={r}, bottom={b}, w={r-l}, h={b-t}")


In [ ]:
# 視覺化：淺綠 lines + 紅色選手區域框
fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(img)
for x in v_lines:
    ax.axvline(x=x, color="#90EE90", linewidth=1.0, alpha=0.7)
for y in h_lines:
    ax.axhline(y=y, color="#90EE90", linewidth=1.0, alpha=0.7)

for i, (l, t, r, b) in enumerate(regions, start=1):
    ax.add_patch(Rectangle((l, t), r-l, b-t, fill=False, edgecolor="red", linewidth=2.5))
    ax.text(l+4, t+18, f"R{i}", color="red", fontsize=12, weight="bold")

ax.set_title("2.2 Athlete regions (red boxes)")
ax.axis("off")
plt.show()


In [ ]:
# 2.3 Build inner cells
cells = []
for (left, top, right, bottom) in regions:
    local_v = [x for x in v_lines if left <= x <= right]
    local_h = [y for y in h_lines if top <= y <= bottom]

    for r1, r2 in zip(local_h, local_h[1:]):
        row_h = r2 - r1
        if row_h < 22 or row_h > 110:
            continue
        for c1, c2 in zip(local_v, local_v[1:]):
            col_w = c2 - c1
            if col_w < 24 or col_w > 170:
                continue
            cells.append((c1 + 3, r1 + 3, c2 - 3, r2 - 3))

cells = sorted(cells, key=lambda c: (c[1], c[0]))
print("detected inner cells:", len(cells))
print("first 10 cells:", cells[:10])


In [ ]:
# 視覺化：淺綠 lines + 紅框 regions + 淺藍小框 cells
fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(img)

for x in v_lines:
    ax.axvline(x=x, color="#90EE90", linewidth=1.0, alpha=0.5)
for y in h_lines:
    ax.axhline(y=y, color="#90EE90", linewidth=1.0, alpha=0.5)

for (l, t, r, b) in regions:
    ax.add_patch(Rectangle((l, t), r-l, b-t, fill=False, edgecolor="red", linewidth=2.0))

for (l, t, r, b) in cells:
    ax.add_patch(Rectangle((l, t), r-l, b-t, fill=False, edgecolor="#87CEFA", linewidth=1.0))  # light blue

ax.set_title("2.3 Inner cells (light blue)")
ax.axis("off")
plt.show()


In [ ]:
# 2.3 + flatten values：確認「要填進 cell 的值順序」
flat_values = renderer._flatten_values(sheet.targets)
print("flatten values:", len(flat_values))
print("first 40 values:")
print(flat_values[:40])

print("\n實際可填數量:", min(len(cells), len(flat_values)))
if len(cells) < len(flat_values):
    print("⚠️ cells 不足，後面的值不會被畫上去。")
elif len(cells) > len(flat_values):
    print("ℹ️ cells 較多，尾端小框會保持空白。")
